# 3.7: Warmup plus decay for LLM pretraining

Most LLM pretraining runs use **linear warmup followed by cosine decay** (or linear decay) to a small final learning rate, often around 10% of the peak or lower. In code:


In [ ]:
import math

def lr_at(step, peak_lr, warmup_steps, total_steps, min_lr_ratio=0.1):
    min_lr = peak_lr * min_lr_ratio
    if step < warmup_steps:
        return peak_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    progress = min(progress, 1.0)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))

# PyTorch: apply via LambdaLR, which multiplies the optimizer's base lr
import torch
model = torch.nn.Linear(10, 10)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda step: lr_at(step, 1.0, warmup_steps=1000, total_steps=100_000))
# In the training loop: loss.backward(); opt.step(); sched.step(); opt.zero_grad()
